# Import lib etc

In [1]:
pip install "kagglehub[pandas-datasets]"

Note: you may need to restart the kernel to use updated packages.


Import dataframe & pick only 3 games

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "kieranpoc/steam-reviews", "weighted_score_above_08.csv")

100%|██████████| 299M/299M [00:09<00:00, 34.4MB/s] 

Extracting zip of weighted_score_above_08.csv...



/opt/anaconda3/lib/python3.12/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [4]:
games = [
    "Counter-Strike 2",
    "PUBG: BATTLEGROUNDS",
    "Grand Theft Auto V"
]

df_3games = df[df["game"].isin(games)].copy()

print(df_3games.shape)
df_3games["game"].value_counts()

(7365, 24)


game
Counter-Strike 2       3017
PUBG: BATTLEGROUNDS    2404
Grand Theft Auto V     1944
Name: count, dtype: int64

# 1. Prepare Gemini Labeling Prompt


In [ ]:
with open("../prompts/codebook_exact.txt", "r") as f:
    SYSTEM = f.read()

GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0

SCHEMA = {
    "type": "object",
    "properties": {
        "harm_mention": {"type": "string", "enum": ["yes", "no", "uncertain"]},
        "harm_reason": {"type": "string"},
        "irony": {"type": "string", "enum": ["yes", "no", "uncertain"]},
        "irony_reason": {"type": "string"},
        "gender_age": {"type": "string", "enum": ["none", "gender", "age", "both", "uncertain"]},
        "gender_age_reason": {"type": "string"}
    },
    "required": [
        "harm_mention", "harm_reason",
        "irony", "irony_reason",
        "gender_age", "gender_age_reason"
    ]
}


# 2. Set Up Gemini API and Labeling Function


In [ ]:
pip install google-genai python-dotenv


In [ ]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def make_payload(row):
    if "recommendation" in row.index:
        recommendation = row["recommendation"]
    elif "voted_up" in row.index:
        recommendation = row["voted_up"]
    else:
        raise KeyError("No recommendation column found")

    if isinstance(recommendation, bool):
        recommendation = "Recommended" if recommendation else "Not recommended"

    return {
        "game": row["game"],
        "review": row["review"],
        "recommendation": recommendation
    }

def label_review(row, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=json.dumps(make_payload(row), ensure_ascii=False),
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM,
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                    response_schema=SCHEMA
                )
            )
            return json.loads(response.text)

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                raise
            time.sleep(30 * (attempt + 1))


# 3. Trial Batch on Human Positive Cases

Test Gemini on 10 reviews where Annotator 1 labeled toxicity or irony as `yes`, then compare the human and Gemini labels.


In [ ]:
annotator_1 = pd.read_csv("../data/blind_labelling_annotator_1.csv")

positive_cases = annotator_1[
    (annotator_1["toxicity_clean"] == "yes") |
    (annotator_1["irony_clean"] == "yes")
].copy()

trial_10 = positive_cases.sample(n=10, random_state=211).copy()

trial_10[["case_id", "game", "review", "recommendation", "toxicity_clean", "irony_clean"]]


In [ ]:
trial_results = []

for _, row in trial_10.iterrows():
    result = label_review(row)

    trial_results.append({
        "case_id": row["case_id"],
        "game": row["game"],
        "review": row["review"],
        "recommendation": row["recommendation"],
        "human_toxicity": row["toxicity_clean"],
        "gemini_toxicity": result["harm_mention"],
        "human_irony": row["irony_clean"],
        "gemini_irony": result["irony"],
        "harm_reason": result["harm_reason"],
        "irony_reason": result["irony_reason"]
    })

    time.sleep(4.2)

trial_results = pd.DataFrame(trial_results)
trial_results["toxicity_match"] = trial_results["human_toxicity"] == trial_results["gemini_toxicity"]
trial_results["irony_match"] = trial_results["human_irony"] == trial_results["gemini_irony"]

trial_results.to_csv("../data/gemini_trial_10.csv", index=False)
trial_results


In [ ]:
print("Toxicity matches:", trial_results["toxicity_match"].sum(), "/", len(trial_results))
print("Irony matches:", trial_results["irony_match"].sum(), "/", len(trial_results))


# 4. Re-run the Same 10 Cases with the Revised Prompt

Reload the revised codebook and run the exact same `trial_10` sample again for a before/after comparison.


In [ ]:
with open("../prompts/codebook_exact.txt", "r") as f:
    SYSTEM = f.read()


In [ ]:
trial_results_revised = []

for _, row in trial_10.iterrows():
    result = label_review(row)

    trial_results_revised.append({
        "case_id": row["case_id"],
        "game": row["game"],
        "review": row["review"],
        "recommendation": row["recommendation"],
        "human_toxicity": row["toxicity_clean"],
        "gemini_toxicity": result["harm_mention"],
        "human_irony": row["irony_clean"],
        "gemini_irony": result["irony"],
        "harm_reason": result["harm_reason"],
        "irony_reason": result["irony_reason"]
    })

    time.sleep(4.2)

trial_results_revised = pd.DataFrame(trial_results_revised)
trial_results_revised["toxicity_match"] = trial_results_revised["human_toxicity"] == trial_results_revised["gemini_toxicity"]
trial_results_revised["irony_match"] = trial_results_revised["human_irony"] == trial_results_revised["gemini_irony"]

trial_results_revised.to_csv("../data/gemini_trial_10_revised_prompt.csv", index=False)
trial_results_revised


In [ ]:
comparison = trial_results[[
    "case_id", "human_toxicity", "gemini_toxicity", "human_irony", "gemini_irony"
]].merge(
    trial_results_revised[[
        "case_id", "gemini_toxicity", "gemini_irony"
    ]],
    on="case_id",
    suffixes=("_before", "_after")
)

display(comparison)
print("Revised toxicity matches:", trial_results_revised["toxicity_match"].sum(), "/", len(trial_results_revised))
print("Revised irony matches:", trial_results_revised["irony_match"].sum(), "/", len(trial_results_revised))
